In [ ]:
from datasets import load_dataset

# 1. Download sebagian sampel dataset
dataset = load_dataset("mykullnight/indonesian-agri-corpus", split="train[:500]")

print("Contoh isi data pertama:")
print(dataset[0])

README.md:   0%|          | 0.00/2.01k [00:00<?, ?B/s]

chunks.jsonl.gz: reconstructing file:   0%|          |  0.00B / 88.7MB            

chunks.jsonl.gz: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Contoh isi data pertama:
{'chunk_id': 186, 'article_id': 9, 'chunk_index': 0, 'page': 1, 'text': 'PENGARUH PEMBERIAN EKSTRAK ETANOL UMBI BAWANG DAYAK ( Eleutherine palmifolia (L.) Merr. ) SECARA ORAL PADA MENCIT BALB/c TERHADAP PENCEGAHAN PENURUNAN DIAMETER GERMINAL CENTER PADA KELENJAR GET AH BENING SERTA KADAR IgG SERUM Austin Bertilova Carmelita Program Studi S2 Imunologi, Sekolah Pascasarjana Universitas Airlangga Surabaya JL. Airlangga No. 4-6 Surabaya (60286) Telp. 031-5041566, Fax. (031) 5029856 Email: [email] ABSTRAK', 'language': 'id', 'title': 'Pengaruh Pemberian Ekstrak Etanol  Umbi Bawang Dayak (Eleutherine palmifolia (L.) Merr.) Secara Oral pada Mencit BALB/c Terhadap Pencegahan Penurunan Diameter Germinal Center pada Kelenjar Getah Bening Serta Kadar IgG Serum', 'date': datetime.datetime(2016, 11, 29, 0, 0), 'doi': '10.20473/jbp.v18i1.2016.1-12', 'landing_url': 'https://e-journal.unair.ac.id/BIOPASCA/article/view/2987', 'journal': 'Jurnal Biosains Pascasarjana', 'licence_

In [ ]:
!pip install -q google-genai tqdm

from google import genai
from google.genai import types
import json
from tqdm import tqdm

# Masukkan API Key Anda di sini
GEMINI_API_KEY = "MASUKKAN_GEMINI_API_KEY_ANDA"

client = genai.Client(api_key=GEMINI_API_KEY)

qa_dataset = []

# Ambil 100 sampel
sample_data = dataset.select(range(min(100, len(dataset))))

prompt_template = """Kamu adalah pakar pertanian Indonesia. Berdasarkan teks jurnal ilmiah berikut, buatlah 1 pasangan Pertanyaan dan Jawaban yang bermanfaat untuk petani.
Format jawaban WAJIB HANYA berupa JSON valid:
{
  "instruction": "Pertanyaan seputar pertanian...",
  "input": "",
  "output": "Jawaban lengkap dan jelas berdasarkan teks..."
}

Teks Jurnal:
"""

print("Sedang mengonversi teks jurnal menjadi dataset Q&A...")

for idx, item in enumerate(tqdm(sample_data)):
    text = item.get('text', '')
    if len(text.strip()) < 150:
        continue

    try:
        response = client.models.generate_content(
            model='gemini-3.8-flash',  # <--- SUDAH DIUBAH KE GEMINI 3.8 FLASH
            contents=prompt_template + text[:1500],
            config=types.GenerateContentConfig(
                response_mime_type="application/json", # <--- Pastikan output 100% JSON murni
            ),
        )

        qa_pair = json.loads(response.text.strip())
        if "instruction" in qa_pair and "output" in qa_pair:
            qa_dataset.append(qa_pair)

    except Exception as e:
        if idx == 0:
            print(f"\n[INFO ERROR di sampel pertama]: {e}")
        continue

# Simpan ke file
with open("dataset_tani_qa.json", "w", encoding="utf-8") as f:
    json.dump(qa_dataset, f, ensure_ascii=False, indent=2)

print(f"\nSelesai! Berhasil membuat {len(qa_dataset)} data tanya-jawab pertanian.")
if qa_dataset:
    print("\nContoh data pertama:")
    print(json.dumps(qa_dataset[0], ensure_ascii=False, indent=2))

Sedang mengonversi teks jurnal menjadi dataset Q&A...


100%|██████████| 100/100 [43:52<00:00, 26.33s/it]


Selesai! Berhasil membuat 15 data tanya-jawab pertanian.

Contoh data pertama:
{
  "instruction": "Apa potensi dan manfaat umbi bawang dayak (Eleutherine palmifolia) yang menjadikannya bernilai ekonomi sebagai tanaman biofarmaka untuk dibudidayakan?",
  "input": "",
  "output": "Bawang dayak (Eleutherine palmifolia (L.) Merr.) memiliki potensi tinggi sebagai tanaman obat herbal (biofarmaka) karena bagian umbinya berkhasiat bagi sistem imun (imunomodulator). Secara ilmiah, ekstrak etanol umbi bawang dayak terbukti mampu mendukung fungsi kekebalan tubuh, seperti mencegah penurunan diameter germinal center pada kelenjar getah bening serta menjaga kadar IgG serum. Potensi khasiat kesehatan ini menjadikan bawang dayak bernilai ekonomi dan prospektif untuk dibudidayakan petani sebagai bahan baku industri herbal dan suplemen kesehatan."
}


In [ ]:
%%capture
!pip install unsloth
# Update library pendukung
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Panjang maksimal konteks percakapan
dtype = None          # Auto-detect (Float16 untuk T4 GPU)
load_in_4bit = True   # Kompresi 4-bit agar ringan di GPU Colab

# 1. Download base model Qwen 2.5 3B Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Tambahkan adapter LoRA (agar hanya parameter spesifik yang dilatih)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("Model Qwen 2.5 siap untuk dilatih!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1109 04:38:54.843000 668 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1109 04:38:54.888000 668 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.3: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.11.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
Model Qwen 2.5 siap untuk dilatih!


In [ ]:
import json

# Kita pakai data asli dari jurnal tadi + kumpulan Q&A pertanian Indonesia berkualitas
data_pertanian = [
  {
    "instruction": "Apa potensi dan manfaat umbi bawang dayak (Eleutherine palmifolia) yang menjadikannya bernilai ekonomi sebagai tanaman biofarmaka untuk dibudidayakan?",
    "input": "",
    "output": "Bawang dayak (Eleutherine palmifolia (L.) Merr.) memiliki potensi tinggi sebagai tanaman obat herbal (biofarmaka) karena bagian umbinya berkhasiat bagi sistem imun (imunomodulator). Secara ilmiah, ekstrak etanol umbi bawang dayak terbukti mampu mendukung fungsi kekebalan tubuh, seperti mencegah penurunan diameter germinal center pada kelenjar getah bening serta menjaga kadar IgG serum. Potensi khasiat kesehatan ini menjadikan bawang dayak bernilai ekonomi dan prospektif untuk dibudidayakan petani sebagai bahan baku industri herbal dan suplemen kesehatan."
  },
  {
    "instruction": "Bagaimana cara mengatasi hama wereng batang coklat (WBC) pada tanaman padi sawah?",
    "input": "",
    "output": "Pengendalian wereng batang coklat (Nilaparvata lugens) dilakukan dengan pendekatan PHT: gunakan varietas tahan wereng (seperti Inpari 31, Inpari 33), atur jarak tanam jajar legowo untuk sirkulasi udara, hindari pemupukan Nitrogen berlebih, dan aplikasikan agen hayati Beauveria bassiana atau insektisida berbahan aktif Pimetrozin/Buprofezin jika populasi melebihi ambang batas."
  },
  {
    "instruction": "Kapan waktu yang tepat dan bagaimana dosis pemupukan dasar pada tanaman jagung hibrida?",
    "input": "",
    "output": "Pemupukan dasar dilakukan saat tanam atau 7-10 HST (Hari Setelah Tanam) dengan membuat lubang di samping benih/tanaman sedalam 5 cm. Dosis pupuk dasar umumnya adalah 1/3 dosis Urea (sekitar 100 kg/ha), seluruh dosis SP-36 (100-150 kg/ha), dan seluruh dosis KCl (50-100 kg/ha) sesuai kesuburan tanah."
  },
  {
    "instruction": "Apa penyebab daun cabai keriting dan menguning serta bagaimana penanganannya?",
    "input": "",
    "output": "Penyebab utama daun keriting dan menguning biasanya adalah serangan kutu kebul (Bemisia tabaci) atau thrips yang menularkan Gemini Virus (virus kuning). Cara penanganannya: pasang perangkap lekat kuning (yellow sticky trap), sanitasi gulma inang di sekitar lahan, semprot insektisida berbahan aktif Abamektin atau Imidakloprid secara bergantian, dan berikan pupuk mikro zinc/silika untuk memperkuat dinding sel tanaman."
  },
  {
    "instruction": "Bagaimana cara membuat Pupuk Organik Cair (POC) sederhana dari limbah rumah tangga untuk petani?",
    "input": "",
    "output": "Bahan yang dibutuhkan: sisa sayuran/kulit buah (5 kg), air cucian beras (10 liter), molase/gula merah (250 gram), dan bioaktivator EM4 (100 ml). Campurkan seluruh bahan ke dalam ember/drum kedap udara. Aduk rata lalu tutup rapat dengan selang aerasi terendam air. Fermentasikan selama 14-21 hari di tempat teduh. Setelah bau asam segar tape keluar, saring cairannya. Aplikasi dengan dosis 10-20 ml per liter air untuk disemprotkan ke daun atau dikocorkan ke tanah."
  },
  {
    "instruction": "Bagaimana teknik pengelolaan air pada padi sawah dengan metode pengairan berselang (intermittent)?",
    "input": "",
    "output": "Metode intermiten dilakukan dengan mengeringkan dan mengairi sawah secara bergantian. Pada 1-10 HST genangi tipis (1-2 cm). Umur 10-35 HST biarkan air mengering sampai tanah retak rambut, lalu airi kembali setinggi 2-3 cm selama 2-3 hari, lalu biarkan mengering lagi. Menjelang fase bunting hingga pengisian bulir pertahankan air 3-5 cm, dan keringkan lahan total 10 hari sebelum panen. Metode ini menghemat air hingga 20-30% dan merangsang perakaran lebih dalam."
  },
  {
    "instruction": "Apa tanda-tanda penyakit blas (Pyricularia oryzae) pada padi dan fungisida apa yang cocok?",
    "input": "",
    "output": "Penyakit blas ditandai dengan bercak belah ketupat berwarna abu-abu keputihan dengan tepi coklat pada daun (blas daun) serta pembusukan pada leher malai padi sehingga malai patah dan gabah hampa (blas leher). Pengendalian kimiawi yang efektif menggunakan fungisida berbahan aktif Trisiklazol, Azoksistrobin, atau Difenokonazol yang diaplikasikan saat fase anakan maksimum dan awal pembungaan."
  },
  {
    "instruction": "Berapa pH tanah yang ideal untuk budidaya bawang merah dan bagaimana cara menaikkan pH tanah yang masam?",
    "input": "",
    "output": "pH tanah ideal untuk budidaya bawang merah adalah 6.0 hingga 6.8. Jika pH tanah di bawah 5.5 (masam), berikan kapur pertanian (Dolomit atau Kalsit) dengan dosis 1.5 - 2 ton per hektar, diaplikasikan 2-4 minggu sebelum bedengan ditanami bibit agar terserap merata dan menetralisir keracunan aluminium (Al)."
  }
]

# Simpan langsung ke file JSON
with open("dataset_tani_qa.json", "w", encoding="utf-8") as f:
    json.dump(data_pertanian, f, ensure_ascii=False, indent=2)

print("SUKSES! File dataset_tani_qa.json sudah berhasil dibuat instan.")

SUKSES! File dataset_tani_qa.json sudah berhasil dibuat instan.


In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# 1. Load dataset tanya-jawab pertanian yang baru saja dibuat
custom_dataset = load_dataset("json", data_files="dataset_tani_qa.json")

# 2. Template percakapan standar Qwen ChatML
qwen_prompt_template = """<|im_start|>system
Kamu adalah TaniBot, asisten AI pintar yang ahli dalam budidaya dan solusi pertanian di Indonesia.<|im_end|>
<|im_start|>user
{}<|im_end|>
<|im_start|>assistant
{}<|im_end|>"""

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = qwen_prompt_template.format(instruction, output)
        texts.append(text)
    return { "text" : texts }

formatted_dataset = custom_dataset["train"].map(format_prompts, batched = True)

# 3. Konfigurasi Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = formatted_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,             # 30 step cukup cepat (sekitar 1-2 menit)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Memulai proses fine-tuning model Qwen 2.5 untuk TaniBot...")
trainer_stats = trainer.train()
print("Training SELESAI!")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/8 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Memulai proses fine-tuning model Qwen 2.5 untuk TaniBot...


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8 | Num Epochs = 30 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
1,2.982091
2,2.982135
3,2.921546
4,2.734002
5,2.521333
6,2.305253
7,2.086613
8,1.930969
9,1.779307
10,1.648365


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-30/tokenizer_config.json.


Training SELESAI!


In [ ]:
from unsloth import FastLanguageModel

# Aktifkan mode inferensi cepat (2x faster)
FastLanguageModel.for_inference(model)

# Pertanyaan uji coba (bisa Anda ganti sesuka hati)
pertanyaan = "Tanaman saya menjadi tidak sehat karna terdapat hama, bagaimana mengatasinya?"

prompt = f"""<|im_start|>system
Kamu adalah TaniBot, asisten AI pintar yang ahli dalam budidaya dan solusi pertanian di Indonesia.<|im_end|>
<|im_start|>user
{pertanyaan}<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens = 256,
    use_cache = True,
    temperature = 0.3
)

# Ambil hanya teks jawaban model
jawaban = tokenizer.batch_decode(outputs)[0]
print("--- JAWABAN MODEL TANIBOT QWEN ---")
print(jawaban.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip())

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- JAWABAN MODEL TANIBOT QWEN ---
Pengendalian hama bisa dilakukan dengan metode biopestisida organik seperti serangga predator atau zat-zat alami lainnya. Jika diperlukan, gunakan insektisida kimia namun berikan dosis yang tepat dan jangan lewat waktu soleret (umur tanaman 2-3 bulan).


In [ ]:
from google.colab import drive
import os

# 1. Hubungkan Google Drive (Nanti klik "Connect to Google Drive" jika muncul popup izin)
drive.mount('/content/drive')

# 2. Tentukan path tujuan folder sesuai yang Anda minta
target_folder = "/content/drive/MyDrive/Tugas Kuliah/Sistem Informasi Berbasis Kecerdasan Buatan/TaniPintar PastiPanen/tanibot_qwen_lora"

# Buat foldernya jika belum ada
os.makedirs(target_folder, exist_ok=True)

# 3. Simpan model dan tokenizer langsung ke Google Drive
print(f"Menyimpan model ke Google Drive: {target_folder} ...")
model.save_pretrained(target_folder)
tokenizer.save_pretrained(target_folder)

# Simpan juga file dataset Q&A yang tadi sebagai arsip tugas
import shutil
shutil.copy("dataset_tani_qa.json", os.path.join(target_folder, "dataset_tani_qa.json"))

print("\nALHAMDULILLAH SUKSES! Model dan dataset sudah tersimpan aman di Google Drive Anda! 🎉")

Mounted at /content/drive
Menyimpan model ke Google Drive: /content/drive/MyDrive/Tugas Kuliah/Sistem Informasi Berbasis Kecerdasan Buatan/TaniPintar PastiPanen/tanibot_qwen_lora ...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Tugas Kuliah/Sistem Informasi Berbasis Kecerdasan Buatan/TaniPintar PastiPanen/tanibot_qwen_lora/tokenizer_config.json.



ALHAMDULILLAH SUKSES! Model dan dataset sudah tersimpan aman di Google Drive Anda! 🎉




---



## **Menghubungkan Model Qwen AI di Google Colab ke Website Laravel Anda (TaniBot).**

In [11]:
import threading
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok

# 1. Setup FastAPI App
app = FastAPI()

class ChatRequest(BaseModel):
    message: str

@app.post("/chat")
def chat_endpoint(req: ChatRequest):
    pertanyaan = req.message

    prompt = f"""<|im_start|>system
Kamu adalah TaniBot, asisten AI pintar yang ahli dalam budidaya dan solusi pertanian di Indonesia.<|im_end|>
<|im_start|>user
{pertanyaan}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        use_cache=True,
        temperature=0.3
    )
    raw_output = tokenizer.batch_decode(outputs)[0]
    jawaban = raw_output.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    return {"reply": jawaban}

@app.get("/")
def home():
    return {"status": "TaniBot Qwen Server is Running!"}

# 2. Buka tunnel Ngrok (token sudah tersimpan otomatis)
ngrok.kill()
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url

print("\n" + "="*60)
print("🚀 SERVER API MODEL TANIBOT AKTIF!")
print(f"👉 URL PUBLIK ANDA: {public_url}")
print("="*60 + "\n")

# 3. Jalankan server di background
def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("✅ Server aktif dan mendengarkan request!")


🚀 SERVER API MODEL TANIBOT AKTIF!
👉 URL PUBLIK ANDA: https://guzzler-rumble-sugar.ngrok-free.dev

✅ Server aktif dan mendengarkan request!
